# KORA Model Training

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run All (Runtime → Run all, or Ctrl+F9)
3. When prompted, paste your **GitHub token** (ghp_…) — it will not appear in output
4. If the session dies mid-training, just **Run All again** — it will resume from the last epoch checkpoint saved to Google Drive

Estimated time: ~25–35 min on T4 GPU

In [ ]:
# Cell 1 — GPU check
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, "
        "then re-run all cells."
    )
print(result.stdout[:600])
print("GPU detected — good to go!")

In [ ]:
# Cell 2 — Mount Google Drive (checkpoints survive session death)
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/kora-checkpoints"
    print(f"Drive mounted. Checkpoints → {CHECKPOINT_DIR}")
except Exception:
    # Fallback for non-Colab environments
    CHECKPOINT_DIR = "/content/kora-checkpoints"
    print(f"Drive not available. Checkpoints → {CHECKPOINT_DIR} (local only)")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# Cell 3 — Clone repository (token never appears in output)
import getpass, subprocess, os

REPO_DIR = "/content/sinonlearning"
BRANCH = "claude/sinon-learning-website-oq7k2i"

if os.path.isdir(REPO_DIR):
    print("Repo already cloned — pulling latest...")
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "origin", BRANCH],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    token = getpass.getpass("GitHub token (ghp_...): ")
    clone_url = f"https://{token}@github.com/sinonymna/sinonlearning.git"
    result = subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        # Mask token in error output before printing
        err = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"Clone failed: {err}")
    print(f"Cloned branch {BRANCH}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 4 — Install pinned dependencies (~3 min)
import subprocess

packages = [
    "torch>=2.2,<2.6",            # 2.6 changed torch.load weights_only default, breaks checkpoint resume
    "transformers>=4.43,<4.47",   # 4.47+ calls accelerator.unwrap_model(keep_torch_compile=False)
    "datasets>=2.20,<3.0",
    "trl>=0.9,<0.12",
    "peft>=0.11,<0.14",
    "accelerate>=0.34,<2.0",
    "bitsandbytes>=0.44",
    "pydantic>=2.7,<3.0",
    "rich>=13.7,<14.0",
    "scikit-learn>=1.5,<2.0",
]

result = subprocess.run(
    ["pip", "install", "-q"] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("Package install failed — see output above.")
print("All packages installed.")

In [ ]:
# Cell 5 — Train (auto-resumes from last checkpoint if one exists)
import os, glob

os.environ["OUTPUT_DIR"] = CHECKPOINT_DIR

# Detect existing checkpoint → set KORA_RESUME so trainer resumes instead of restarting
checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")))
if checkpoints:
    latest = checkpoints[-1]
    os.environ["KORA_RESUME"] = latest
    print(f"Resuming from checkpoint: {latest}")
else:
    os.environ.pop("KORA_RESUME", None)
    print("No checkpoint found — starting from scratch.")

!python kora_model/train_kora.py train

In [ ]:
# Cell 6 — Evaluate the trained adapter
import os
os.environ["OUTPUT_DIR"] = CHECKPOINT_DIR

!python kora_model/train_kora.py eval

In [ ]:
# Cell 7 — Zip and download the adapter
import os, subprocess

ZIP_PATH = "/content/kora-adapter.zip"

result = subprocess.run(
    ["zip", "-r", ZIP_PATH, CHECKPOINT_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Zip failed.")

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Adapter zipped: {ZIP_PATH} ({size_mb:.1f} MB)")

try:
    from google.colab import files
    files.download(ZIP_PATH)
    print("Download started — check your browser.")
except Exception:
    print(f"Not in Colab — adapter is at: {ZIP_PATH}")